# Testing
In this notebook, we do some minimal benchmarking of the DMPNN implementation on a synthetically generated dataset and compare to the GINEConv model. 

In [1]:
import os
import random
import time

import pandas as pd
import torch

os.chdir("..")

from dmpnn.synthetic import generate_unique_graphs
from dmpnn.model import DMPNNRegressor
from dmpnn.training import DMPNNTrainer


In [3]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
SEEDS = [0, 1, 2]
BATCH_SIZE = 32
EPOCHS = 50
LR = 1e-3

train_graphs, train_sigs = generate_unique_graphs(10000, min_nodes=3, max_nodes=10, seed=42)
val_graphs, _ = generate_unique_graphs(2000, min_nodes=3, max_nodes=10, existing_sigs=train_sigs, seed=43)
test_graphs, _ = generate_unique_graphs(2000, min_nodes=3, max_nodes=10, existing_sigs=train_sigs, seed=44)
y_true_test = torch.vstack([g["y"] for g in test_graphs])

print(f"Using device: {device}")
print(f"Train/val/test sizes: {len(train_graphs)}, {len(val_graphs)}, {len(test_graphs)}")


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


Using device: mps
Train/val/test sizes: 10000, 2000, 2000


In [13]:
DMPNN_CONFIG = {
    "node_feat_dim": 4,
    "edge_feat_dim": 2,
    "hidden_dim": 128,
    "num_steps": 3,
    "head_hidden_size": 128,
    "output_size": 1,
}


def run_dmpnn_seed(seed):
    set_seed(seed)

    model = DMPNNRegressor(**DMPNN_CONFIG)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    trainer = DMPNNTrainer(
        model,
        optimizer=optimizer,
        loss_fn=torch.nn.MSELoss(),
        device=device,
    )

    start = time.perf_counter()
    trainer.fit(
        train_graphs,
        val_graphs,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=False,
    )
    fit_seconds = time.perf_counter() - start

    test_mse = trainer.evaluate(test_graphs, batch_size=BATCH_SIZE)
    test_preds = trainer.predict(test_graphs, batch_size=BATCH_SIZE)
    test_mae = torch.mean(torch.abs(test_preds - y_true_test)).item()

    return {
        "model": "DMPNN",
        "seed": seed,
        "test_mse": test_mse,
        "test_mae": test_mae,
        "fit_seconds": fit_seconds,
        "param_count": count_parameters(trainer.model),
    }


dmpnn_results = [run_dmpnn_seed(seed) for seed in SEEDS]
pd.DataFrame(dmpnn_results)


,model,seed,test_mse,test_mae,fit_seconds,param_count
0,DMPNN,0,0.002763,0.038737,114.203657,67201
1,DMPNN,1,0.004692,0.050760,90.485214,67201
2,DMPNN,2,0.005596,0.055051,80.367894,67201


In [14]:
import torch.nn as nn

try:
    from torch_geometric.data import Data
    from torch_geometric.loader import DataLoader
    from torch_geometric.nn import GINEConv, global_add_pool
except ImportError as e:
    raise ImportError("Install torch_geometric to run the GINE benchmark cell.") from e


def to_pyg_data(graph):
    return Data(
        x=graph["X"],
        edge_index=graph["edge_index"],
        edge_attr=graph["B"],
        y=graph["y"],
    )


train_dataset = [to_pyg_data(g) for g in train_graphs]
val_dataset = [to_pyg_data(g) for g in val_graphs]
test_dataset = [to_pyg_data(g) for g in test_graphs]


class GINERegressor(nn.Module):
    def __init__(self, in_channels, edge_dim, hidden_channels, num_layers, out_channels=1, dropout=0.0):
        super().__init__()
        self.node_encoder = nn.Linear(in_channels, hidden_channels)
        self.convs = nn.ModuleList()
        self.dropout = nn.Dropout(dropout)

        for _ in range(num_layers):
            mlp = nn.Sequential(
                nn.Linear(hidden_channels, hidden_channels),
                nn.ReLU(),
                nn.Linear(hidden_channels, hidden_channels),
            )
            self.convs.append(GINEConv(mlp, edge_dim=edge_dim))

        self.head = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, out_channels),
        )

    def forward(self, batch):
        x, edge_index = batch.x, batch.edge_index
        edge_attr, batch_vec = batch.edge_attr, batch.batch

        x = self.node_encoder(x)
        for conv in self.convs:
            x = conv(x, edge_index, edge_attr)
            x = torch.relu(x)
            x = self.dropout(x)

        g = global_add_pool(x, batch_vec)
        return self.head(g)


GINE_CONFIG = {
    "in_channels": 4,
    "edge_dim": 2,
    "hidden_channels": 128,
    "num_layers": 3,
    "out_channels": 1,
    "dropout": 0.0,
}


def train_epoch_gine(model, loader, optimizer, loss_fn):
    model.train()
    total_loss = 0.0
    total_graphs = 0

    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        y_hat = model(batch)
        y = batch.y.view(-1, 1)
        loss = loss_fn(y_hat, y)
        loss.backward()
        optimizer.step()

        batch_graphs = y.shape[0]
        total_loss += loss.item() * batch_graphs
        total_graphs += batch_graphs

    return total_loss / total_graphs


@torch.no_grad()
def evaluate_gine(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    total_graphs = 0
    preds = []
    targets = []

    for batch in loader:
        batch = batch.to(device)
        y_hat = model(batch)
        y = batch.y.view(-1, 1)
        loss = loss_fn(y_hat, y)

        batch_graphs = y.shape[0]
        total_loss += loss.item() * batch_graphs
        total_graphs += batch_graphs
        preds.append(y_hat.cpu())
        targets.append(y.cpu())

    avg_loss = total_loss / total_graphs
    preds = torch.cat(preds, dim=0)
    targets = torch.cat(targets, dim=0)
    mae = torch.mean(torch.abs(preds - targets)).item()
    return avg_loss, mae, preds, targets


def run_gine_seed(seed):
    set_seed(seed)

    model = GINERegressor(**GINE_CONFIG).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = torch.nn.MSELoss()

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    start = time.perf_counter()
    for _ in range(EPOCHS):
        train_epoch_gine(model, train_loader, optimizer, loss_fn)
        evaluate_gine(model, val_loader, loss_fn)
    fit_seconds = time.perf_counter() - start

    test_mse, test_mae, _, _ = evaluate_gine(model, test_loader, loss_fn)

    return {
        "model": "GINE",
        "seed": seed,
        "test_mse": test_mse,
        "test_mae": test_mae,
        "fit_seconds": fit_seconds,
        "param_count": count_parameters(model),
    }


gine_results = [run_gine_seed(seed) for seed in SEEDS]
pd.DataFrame(gine_results)


,model,seed,test_mse,test_mae,fit_seconds,param_count
0,GINE,0,0.004309,0.053137,150.723464,117505
1,GINE,1,0.001865,0.031621,145.683543,117505
2,GINE,2,0.031438,0.143864,136.424167,117505


In [15]:
all_results = pd.concat(
    [pd.DataFrame(dmpnn_results), pd.DataFrame(gine_results)],
    ignore_index=True,
)

summary = all_results.groupby("model", as_index=False).agg(
    test_mse_mean=("test_mse", "mean"),
    test_mse_std=("test_mse", "std"),
    test_mae_mean=("test_mae", "mean"),
    test_mae_std=("test_mae", "std"),
    fit_seconds_mean=("fit_seconds", "mean"),
    fit_seconds_std=("fit_seconds", "std"),
    param_count=("param_count", "first"),
)

print("Per-seed results:")
display(all_results)

print("Summary across 3 seeds:")
display(summary.sort_values("test_mse_mean"))


Per-seed results:


,model,seed,test_mse,test_mae,fit_seconds,param_count
0,DMPNN,0,0.002763,0.038737,114.203657,67201
1,DMPNN,1,0.004692,0.050760,90.485214,67201
2,DMPNN,2,0.005596,0.055051,80.367894,67201
3,GINE,0,0.004309,0.053137,150.723464,117505
4,GINE,1,0.001865,0.031621,145.683543,117505
5,GINE,2,0.031438,0.143864,136.424167,117505


Summary across 3 seeds:


,model,test_mse_mean,test_mse_std,test_mae_mean,test_mae_std,fit_seconds_mean,fit_seconds_std,param_count
0,DMPNN,0.004350,0.001447,0.048183,0.008457,95.018922,17.367516,67201
1,GINE,0.012537,0.016414,0.076207,0.059572,144.277058,7.252663,117505


In [ ]:
from dmpnn.adapters import from_pyg_dataset
from torch_geometric.datasets import MoleculeNet
import numpy as np

# Use a standard molecular regression benchmark from PyG.
dataset = MoleculeNet(root="data/MoleculeNet", name="ESOL")

graphs = from_pyg_dataset(dataset)

num_graphs = len(graphs)
train_end = int(0.8 * num_graphs)
val_end = int(0.9 * num_graphs)

train_graphs = graphs[:train_end]
val_graphs = graphs[train_end:val_end]
test_graphs = graphs[val_end:]

node_feat_dim = train_graphs[0]["X"].shape[1]
edge_feat_dim = train_graphs[0]["B"].shape[1]

model = DMPNNRegressor(
    node_feat_dim=node_feat_dim,
    edge_feat_dim=edge_feat_dim,
    hidden_dim=128,
    num_steps=3,
    head_hidden_size=128,
    output_size=1,
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
trainer = DMPNNTrainer(
    model,
    optimizer=optimizer,
    loss_fn=torch.nn.MSELoss(),
    device=device,
)

history = trainer.fit(
    train_graphs,
    val_graphs,
    epochs=20,
    batch_size=32,
    verbose=True,
)

preds = trainer.predict(test_graphs, batch_size=32)
y_true = torch.vstack([g["y"] for g in test_graphs])
test_mse = trainer.evaluate(test_graphs, batch_size=32)
test_mae = torch.mean(torch.abs(preds - y_true)).item()
test_rmse = np.sqrt(test_mse)

print("\nPyG example dataset: ESOL")
print(f"Train/val/test")
from dmpnn.adapters import from_pyg_dataset
from torch_geometric.datasets import MoleculeNet
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINEConv, global_add_pool
import torch.nn as nn
import numpy as np

# Use a standard molecular regression benchmark from PyG.
dataset = MoleculeNet(root="data/MoleculeNet", name="ESOL")
graphs = from_pyg_dataset(dataset)

num_graphs = len(graphs)
train_end = int(0.8 * num_graphs)
val_end = int(0.9 * num_graphs)

train_graphs = graphs[:train_end]
val_graphs = graphs[train_end:val_end]
test_graphs = graphs[val_end:]

node_feat_dim = train_graphs[0]["X"].shape[1]
edge_feat_dim = train_graphs[0]["B"].shape[1]

# ---------------------------
# DMPNN
# ---------------------------
model = DMPNNRegressor(
    node_feat_dim=node_feat_dim,
    edge_feat_dim=edge_feat_dim,
    hidden_dim=128,
    num_steps=3,
    head_hidden_size=128,
    output_size=1,
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
trainer = DMPNNTrainer(
    model,
    optimizer=optimizer,
    loss_fn=torch.nn.MSELoss(),
    device=device,
)

history = trainer.fit(
    train_graphs,
    val_graphs,
    epochs=20,
    batch_size=32,
    verbose=True,
)

preds = trainer.predict(test_graphs, batch_size=32)
y_true = torch.vstack([g["y"] for g in test_graphs])
test_mse = trainer.evaluate(test_graphs, batch_size=32)
test_mae = torch.mean(torch.abs(preds - y_true)).item()
test_rmse = np.sqrt(test_mse)

print("\nDMPNN on ESOL")
print(f"Train/val/test sizes: {len(train_graphs)}, {len(val_graphs)}, {len(test_graphs)}")
print(f"Test MSE: {test_mse:.4f}")
print(f"Test RMSE: {test_rmse:.4f}")
print(f"Test MAE: {test_mae:.4f}")
print("First 10 DMPNN predictions:")
for i, (y_t, y_p) in enumerate(zip(y_true[:10], preds[:10])):
    print(f"  graph {i:>2}: true = {y_t.item():>6.2f} | pred = {y_p.item():>6.2f}")

# ---------------------------
# GINEConv baseline
# ---------------------------
def to_pyg_data(graph):
    return Data(
        x=graph["X"],
        edge_index=graph["edge_index"],
        edge_attr=graph["B"],
        y=graph["y"],
    )

train_dataset = [to_pyg_data(g) for g in train_graphs]
val_dataset = [to_pyg_data(g) for g in val_graphs]
test_dataset = [to_pyg_data(g) for g in test_graphs]

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


class GINERegressor(nn.Module):
    def __init__(self, in_channels, edge_dim, hidden_channels, num_layers, out_channels=1, dropout=0.0):
        super().__init__()
        self.node_encoder = nn.Linear(in_channels, hidden_channels)
        self.convs = nn.ModuleList()
        self.dropout = nn.Dropout(dropout)

        for _ in range(num_layers):
            mlp = nn.Sequential(
                nn.Linear(hidden_channels, hidden_channels),
                nn.ReLU(),
                nn.Linear(hidden_channels, hidden_channels),
            )
            self.convs.append(GINEConv(mlp, edge_dim=edge_dim))

        self.head = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, out_channels),
        )

    def forward(self, batch):
        x, edge_index, edge_attr, batch_vec = batch.x, batch.edge_index, batch.edge_attr, batch.batch

        x = self.node_encoder(x)
        for conv in self.convs:
            x = conv(x, edge_index, edge_attr)
            x = torch.relu(x)
            x = self.dropout(x)

        g = global_add_pool(x, batch_vec)
        return self.head(g)


gine_model = GINERegressor(
    in_channels=node_feat_dim,
    edge_dim=edge_feat_dim,
    hidden_channels=128,
    num_layers=3,
    out_channels=1,
    dropout=0.0,
).to(device)

gine_optimizer = torch.optim.Adam(gine_model.parameters(), lr=1e-3)
loss_fn = torch.nn.MSELoss()


def train_epoch_gine(loader):
    gine_model.train()
    total_loss = 0.0
    total_graphs = 0

    for batch in loader:
        batch = batch.to(device)
        gine_optimizer.zero_grad()
        y_hat = gine_model(batch)
        y = batch.y.view(-1, 1)
        loss = loss_fn(y_hat, y)
        loss.backward()
        gine_optimizer.step()

        batch_graphs = y.shape[0]
        total_loss += loss.item() * batch_graphs
        total_graphs += batch_graphs

    return total_loss / total_graphs


@torch.no_grad()
def evaluate_gine(loader):
    gine_model.eval()
    total_loss = 0.0
    total_graphs = 0
    preds = []
    targets = []

    for batch in loader:
        batch = batch.to(device)
        y_hat = gine_model(batch)
        y = batch.y.view(-1, 1)
        loss = loss_fn(y_hat, y)

        batch_graphs = y.shape[0]
        total_loss += loss.item() * batch_graphs
        total_graphs += batch_graphs

        preds.append(y_hat.cpu())
        targets.append(y.cpu())

    avg_loss = total_loss / total_graphs
    preds = torch.cat(preds, dim=0)
    targets = torch.cat(targets, dim=0)
    mae = torch.mean(torch.abs(preds - targets)).item()
    rmse = np.sqrt(avg_loss)
    return avg_loss, rmse, mae, preds, targets


for epoch in range(20):
    train_loss = train_epoch_gine(train_loader)
    val_mse, val_rmse, val_mae, _, _ = evaluate_gine(val_loader)
    print(
        f"Epoch {epoch + 1}/20 | "
        f"GINE train_loss={train_loss:.4f} | "
        f"val_mse={val_mse:.4f} | val_rmse={val_rmse:.4f} | val_mae={val_mae:.4f}"
    )

gine_test_mse, gine_test_rmse, gine_test_mae, gine_preds, gine_targets = evaluate_gine(test_loader)

print("\nGINEConv on ESOL")
print(f"Test MSE: {gine_test_mse:.4f}")
print(f"Test RMSE: {gine_test_rmse:.4f}")
print(f"Test MAE: {gine_test_mae:.4f}")
print("First 10 GINE predictions:")
for i, (y_t, y_p) in enumerate(zip(gine_targets[:10], gine_preds[:10])):
    print(f"  graph {i:>2}: true = {y_t.item():>6.2f} | pred = {y_p.item():>6.2f}")
print(f"sizes: {len(train_graphs)}, {len(val_graphs)}, {len(test_graphs)}")
print("First 10 predictions:")
for i, (y_t, y_p) in enumerate(zip(y_true[:10], preds[:10])):
    print(f"  graph {i:>2}: true = {y_t.item():>6.2f} | pred = {y_p.item():>6.2f}")


Epoch 1/20 | train_loss=8.9769 | val_loss=2.5838
Epoch 2/20 | train_loss=2.0816 | val_loss=1.9558
Epoch 3/20 | train_loss=1.7770 | val_loss=1.8252
Epoch 4/20 | train_loss=1.6527 | val_loss=1.5853
Epoch 5/20 | train_loss=1.4230 | val_loss=2.2217
Epoch 6/20 | train_loss=1.6977 | val_loss=1.7408
Epoch 7/20 | train_loss=1.3077 | val_loss=1.6394
Epoch 8/20 | train_loss=1.1594 | val_loss=1.7875
Epoch 9/20 | train_loss=1.5932 | val_loss=1.7670
Epoch 10/20 | train_loss=1.2675 | val_loss=1.9470
Epoch 11/20 | train_loss=1.2554 | val_loss=1.5769
Epoch 12/20 | train_loss=1.1359 | val_loss=1.2683
Epoch 13/20 | train_loss=1.1638 | val_loss=1.6736
Epoch 14/20 | train_loss=1.0185 | val_loss=1.3472
Epoch 15/20 | train_loss=0.8691 | val_loss=1.4245
Epoch 16/20 | train_loss=0.9745 | val_loss=1.4659
Epoch 17/20 | train_loss=0.9181 | val_loss=1.2499
Epoch 18/20 | train_loss=0.9915 | val_loss=1.3877
Epoch 19/20 | train_loss=0.8959 | val_loss=1.1597
Epoch 20/20 | train_loss=0.9519 | val_loss=1.5819

PyG exam

In [4]:
from dmpnn.adapters import from_pyg_dataset
from torch_geometric.datasets import MoleculeNet

# Example: binary molecular classification with a standard PyG benchmark dataset.
# BBBP labels are treated as 0/1 binary targets for graph-level classification.

bbbp_dataset = MoleculeNet(root="data/MoleculeNet", name="BBBP")
bbbp_graphs = from_pyg_dataset(bbbp_dataset)

clean_graphs = []
for g in bbbp_graphs:
    y = g["y"]
    if not torch.isfinite(y).all():
        continue
    g = dict(g)
    g["y"] = (y > 0).float()
    clean_graphs.append(g)

num_graphs = len(clean_graphs)
train_end = int(0.8 * num_graphs)
val_end = int(0.9 * num_graphs)

train_graphs = clean_graphs[:train_end]
val_graphs = clean_graphs[train_end:val_end]
test_graphs = clean_graphs[val_end:]

node_feat_dim = train_graphs[0]["X"].shape[1]
edge_feat_dim = train_graphs[0]["B"].shape[1]

clf_model = DMPNNRegressor(
    node_feat_dim=node_feat_dim,
    edge_feat_dim=edge_feat_dim,
    hidden_dim=128,
    num_steps=3,
    head_hidden_size=128,
    output_size=1,
)

clf_optimizer = torch.optim.Adam(clf_model.parameters(), lr=1e-3)
clf_trainer = DMPNNTrainer(
    clf_model,
    optimizer=clf_optimizer,
    loss_fn=torch.nn.BCEWithLogitsLoss(),
    device=device,
    task_type="binary_classification",
)

clf_history = clf_trainer.fit(
    train_graphs,
    val_graphs,
    epochs=10,
    batch_size=32,
    verbose=True,
)

# `predict()` returns hard 0/1 labels for binary classification.
pred_labels = clf_trainer.predict(test_graphs, batch_size=32).float()
y_true = torch.vstack([g["y"] for g in test_graphs]).float()

test_bce = clf_trainer.evaluate(test_graphs, batch_size=32)
accuracy = (pred_labels.view_as(y_true) == y_true).float().mean().item()
positive_rate = pred_labels.float().mean().item()

print("PyG binary classification example: BBBP")
print(f"Train/val/test sizes: {len(train_graphs)}, {len(val_graphs)}, {len(test_graphs)}")
print(f"Test BCE loss: {test_bce:.4f}")
print(f"Test accuracy: {accuracy:.4f}")
print(f"Predicted positive rate: {positive_rate:.4f}")
print("First 10 binary predictions:")
for i, (y_t, y_p) in enumerate(zip(y_true[:10], pred_labels[:10])):
    print(f"  graph {i:>2}: true = {int(y_t.item())} | pred = {int(y_p.item())}")



Epoch 1/10 | train_loss=0.8211 | val_loss=0.1661
Epoch 2/10 | train_loss=0.5885 | val_loss=0.4817
Epoch 3/10 | train_loss=0.5324 | val_loss=0.7302
Epoch 4/10 | train_loss=0.5098 | val_loss=0.4391
Epoch 5/10 | train_loss=0.5098 | val_loss=0.4633
Epoch 6/10 | train_loss=0.4810 | val_loss=0.2158
Epoch 7/10 | train_loss=0.4832 | val_loss=0.2363
Epoch 8/10 | train_loss=0.4475 | val_loss=0.2258
Epoch 9/10 | train_loss=0.4329 | val_loss=0.3967
Epoch 10/10 | train_loss=0.4298 | val_loss=0.3015
PyG binary classification example: BBBP
Train/val/test sizes: 1631, 204, 204
Test BCE loss: 0.2965
Test accuracy: 0.8971
Predicted positive rate: 0.8971
First 10 binary predictions:
  graph  0: true = 1 | pred = 1
  graph  1: true = 1 | pred = 1
  graph  2: true = 1 | pred = 1
  graph  3: true = 1 | pred = 1
  graph  4: true = 1 | pred = 1
  graph  5: true = 1 | pred = 1
  graph  6: true = 1 | pred = 1
  graph  7: true = 1 | pred = 1
  graph  8: true = 1 | pred = 1
  graph  9: true = 1 | pred = 1
